<a href="https://colab.research.google.com/github/KDK-00/deeplearning/blob/main/news_api_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **News 추천 알고리즘**

사용 전 Newsapi.org 홈페이지에 접속하여

개인 api key를 발급받아야 합니다.



In [25]:
!pip install requests

해당하는 부분에 발급받은 api key를 입력해주세요

In [26]:
import requests

# API 키 설정
API_KEY = '630646c0b54e41abb999fe00886bfe2c'
BASE_URL = 'https://newsapi.org/v2/'

In [27]:
def get_top_headlines(api_key, country='kr', category='general', page_size=10):
    url = f"{BASE_URL}top-headlines"  # BASE_URL에 이미 '/v2/'가 포함되어 있음
    params = {
        'apiKey': api_key,
        'country': country,
        'category': category,
        'pageSize': page_size
    }
    response = requests.get(url, params=params)

    # Check for HTTP errors
    response.raise_for_status()  # Raise an exception for bad status codes

    return response.json()

# 한국의 비즈니스 뉴스 헤드라인 가져오기
headlines = get_top_headlines(API_KEY)

# 결과 출력
for article in headlines['articles']:
    print(f"Title: {article['title']}")
    print(f"Description: {article['description']}")
    print(f"URL: {article['url']}")
    print("-" * 50)

Title: 메트로바니아풍 소울라이크 '더 라스트 페이스’ 실물 패키지 발매 - 인벤
Description: ㈜에이치투 인터렉티브(이하 H2 INTERACTIVE, 대표 허준하)는 쿠미 소울 게임즈(Kumi Souls Games)의 메트로바니아풍 소울라이크 게임 ‘더 라스트 페이스(The Last Faith)’의 실물 패키지인 ‘더 라스트 페이스:..
URL: https://www.inven.co.kr/webzine/news/?news=297219
--------------------------------------------------
Title: 손웅정 감독 '아동학대 논란'에 시민단체 토론회…"본질은 폭력" - 한국경제
Description: 손웅정 감독 '아동학대 논란'에 시민단체 토론회…"본질은 폭력", 사회
URL: https://www.hankyung.com/article/202407042434Y
--------------------------------------------------
Title: 삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스
Description: 삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼성전자 디바이스솔루션(DS) 부문은 오늘(4일) HBM 개발팀 신설을 골자로 하는 조직 개편을 실시했습니다.<br /> <br />메모리사업부 내에 있었던 HBM 전담 조직을 팀으로 신설한 것으로, 신임 팀장은 고성능 D램 제품 설계 전문가인 손영수 부사장이 맡기로 했습니다.<br /> <br />이 같은 HBM 개발팀 신설은 인공지능 시장 확대로 HBM 수요가 급증하면서, HB…
URL: https://news.kbs.co.kr/news/view.do?ncd=8003727
--------------------------------------------------
Title: 암 진단 후 이렇게 식사한 사람들, 더 오래 산다 - 동아일보
Description: 지중해식 

In [28]:
def get_top_headlines(api_key, country='kr', category='business', page_size=100, max_pages=5):
    url = f"{BASE_URL}top-headlines"
    all_articles = []
    for page in range(1, max_pages + 1):
        params = {
            'apiKey': api_key,
            'country': country,
            'category': category,
            'pageSize': page_size,
            'page': page
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        all_articles.extend(data['articles'])
        if len(data['articles']) < page_size:
            break  # 더 이상 기사가 없으면 중단
    return all_articles

# 한국의 비즈니스 뉴스 헤드라인 많이 가져오기
articles = get_top_headlines(API_KEY)

# 데이터 개수 확인
print(f"Total articles fetched: {len(articles)}")

Total articles fetched: 70


In [29]:
import re
import pandas as pd

def preprocess_articles(articles):
    cleaned_articles = []
    for article in articles:
        title = article['title'] if article['title'] else ''
        description = article['description'] if article['description'] else ''
        content = f"{title} {description}"
        content = re.sub(r'<[^>]+>', '', content)
        content = re.sub(r'[^a-zA-Z0-9가-힣\s]', '', content)
        cleaned_articles.append({
            'content': content.strip(),
            'source': article['source']['name'],
            'publishedAt': article['publishedAt']
        })
    return cleaned_articles

cleaned_articles = preprocess_articles(articles)

# DataFrame으로 정리
df = pd.DataFrame(cleaned_articles, columns=['content'])
print(df.head())

                                             content
0  삼성전자 조직 개편 HBM 개발팀 신설HBM에 더 집중  KBS뉴스 삼성전자가 HB...
1  골드만삭스 한은 8월 금리 인하할 것하반기 물가 20로 둔화  연합인포맥스 유력 투...
2  10명한테 물어놓고 10명 중 9명 합격에듀윌 과태료 500만원  한겨레 공무원 온...
3  서울 기후동행카드에 남양주시 참여8월 별내진접선 적용  연합뉴스 서울연합뉴스 김기훈...
4  빅테크 꼭지 신호인가젠슨 황 이어 베이조스도 주식 매각 나선다  매일경제 최근 신고...


In [30]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import linear_kernel

# def build_recommendation_model(cleaned_articles):
#     tfidf = TfidfVectorizer(stop_words='english')
#     tfidf_matrix = tfidf.fit_transform(cleaned_articles)
#     cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
#     return cosine_sim

# # 추천 모델 구축
# cosine_sim = build_recommendation_model(cleaned_articles)

# def recommend_articles(index, cosine_sim, df, top_n=5):
#     sim_scores = list(enumerate(cosine_sim[index]))
#     sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
#     sim_scores = sim_scores[1:top_n + 1]

#     article_indices = [i[0] for i in sim_scores]
#     return df.iloc[article_indices]

# # 첫 번째 기사와 유사한 기사 추천
# recommended_articles = recommend_articles(0, cosine_sim, df)
# print(recommended_articles)

In [31]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import json

# **json형식으로 API 가져오기**

In [32]:
import requests
url = ('https://newsapi.org/v2/top-headlines?'
       'country=kr&'
       'category=business&'
       'apiKey=630646c0b54e41abb999fe00886bfe2c')
response = requests.get(url)
print(response.json())

{'status': 'ok', 'totalResults': 70, 'articles': [{'source': {'id': None, 'name': 'Kbs.co.kr'}, 'author': '김지숙', 'title': "삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스", 'description': '삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼성전자 디바이스솔루션(DS) 부문은 오늘(4일) HBM 개발팀 신설을 골자로 하는 조직 개편을 실시했습니다.<br /> <br />메모리사업부 내에 있었던 HBM 전담 조직을 팀으로 신설한 것으로, 신임 팀장은 고성능 D램 제품 설계 전문가인 손영수 부사장이 맡기로 했습니다.<br /> <br />이 같은 HBM 개발팀 신설은 인공지능 시장 확대로 HBM 수요가 급증하면서, HB…', 'url': 'https://news.kbs.co.kr/news/view.do?ncd=8003727', 'urlToImage': 'http://news.kbs.co.kr/data/news/2024/07/04/20240704_qmlaaV.jpg', 'publishedAt': '2024-07-04T06:20:00Z', 'content': "HBM( ) .(DS) (4) HBM .\r\nHBM , D .\r\nHBM HBM , HBM .\r\nHBM3 HBM3E HBM4 .\r\n(AVP) .\r\nAVP AVP , .\r\n5 .\r\n.\r\n[ : / ]\r\n: 'KBS' , : 02-781-1234, 4444: kbs1234@kbs.co.kr, , KBS !"}, {'source': {'id': None, 'name': 'Einfomax.co.kr'}, 'author': '정윤교 기자', 'title': '골드만삭스 "한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화" - 연합인포맥스', 'description': '유력 투자은행 골드만삭스는 한국은행이 올해 8월

In [33]:
json.loads(response.text)

{'status': 'ok',
 'totalResults': 70,
 'articles': [{'source': {'id': None, 'name': 'Kbs.co.kr'},
   'author': '김지숙',
   'title': "삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스",
   'description': '삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼성전자 디바이스솔루션(DS) 부문은 오늘(4일) HBM 개발팀 신설을 골자로 하는 조직 개편을 실시했습니다.<br /> <br />메모리사업부 내에 있었던 HBM 전담 조직을 팀으로 신설한 것으로, 신임 팀장은 고성능 D램 제품 설계 전문가인 손영수 부사장이 맡기로 했습니다.<br /> <br />이 같은 HBM 개발팀 신설은 인공지능 시장 확대로 HBM 수요가 급증하면서, HB…',
   'url': 'https://news.kbs.co.kr/news/view.do?ncd=8003727',
   'urlToImage': 'http://news.kbs.co.kr/data/news/2024/07/04/20240704_qmlaaV.jpg',
   'publishedAt': '2024-07-04T06:20:00Z',
   'content': "HBM( ) .(DS) (4) HBM .\r\nHBM , D .\r\nHBM HBM , HBM .\r\nHBM3 HBM3E HBM4 .\r\n(AVP) .\r\nAVP AVP , .\r\n5 .\r\n.\r\n[ : / ]\r\n: 'KBS' , : 02-781-1234, 4444: kbs1234@kbs.co.kr, , KBS !"},
  {'source': {'id': None, 'name': 'Einfomax.co.kr'},
   'author': '정윤교 기자',
   'title': '골드만삭스 "한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화" - 연합인포맥스',
   'descrip

In [34]:
json_data = pd.json_normalize(json.loads(response.text)['articles'])
json_data

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,김지숙,"삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스",삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼...,https://news.kbs.co.kr/news/view.do?ncd=8003727,http://news.kbs.co.kr/data/news/2024/07/04/202...,2024-07-04T06:20:00Z,"HBM( ) .(DS) (4) HBM .\r\nHBM , D .\r\nHBM HBM...",None,Kbs.co.kr
1,정윤교 기자,"골드만삭스 ""한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화"" - 연합인포맥스",유력 투자은행 골드만삭스는 한국은행이 올해 8월 기준금리 인하에 나설 것으로 전망했...,https://news.einfomax.co.kr/news/articleView.h...,https://cdn.news.einfomax.co.kr/news/thumbnail...,2024-07-04T05:57:35Z,[ . DB ]\r\n(=) = 8 . 2.0% .\r\n3() 11 ( ) .\r...,None,Einfomax.co.kr
2,안태호,10명한테 물어놓고 “10명 중 9명 합격”…에듀윌 과태료 500만원 - 한겨레,공무원 온라인 강의 서비스를 제공하는 에듀윌이 단 10명만 응답한 설문 결과를 근거...,https://www.hani.co.kr/arti/economy/economy_ge...,https://flexible.img.hani.co.kr/flexible/norma...,2024-07-04T05:22:00Z,None,None,Hani.co.kr
3,김기훈,서울 기후동행카드에 남양주시 참여…8월 별내·진접선 적용 - 연합뉴스,(서울=연합뉴스) 김기훈 기자 = 8월 지하철 8호선의 연장 별내선 개통에 맞춰 별...,https://www.yna.co.kr/view/AKR20240704018700004,https://img4.yna.co.kr/photo/yna/YH/2024/07/04...,2024-07-04T05:10:00Z,- (=) 4 - . 2024.7.4 [ . DB ] photo@yna.co.kr\...,None,Yna.co.kr
4,이덕주,“빅테크 꼭지 신호인가?”...젠슨 황 이어 베이조스도 주식 매각 나선다 - 매일경제,최근 신고가를 경신하고 있는 주요 빅테크 기업들의 창업자들이 주식을 매각하면서 주가...,https://www.mk.co.kr/news/business/11058688,https://wimg-orig.mk.co.kr/news/cms/202407/04/...,2024-07-04T04:53:51Z,.\r\n3() CEO 130 (SEC) . 130 16900( 2300) .\r\...,None,Mk.co.kr
5,떤 풍(Tan phung) 기자,"베트남 총리, 삼성전자 평택캠퍼스 방문…방한일정 마무리 - 인사이드비나","[인사이드비나=하노이, 떤 풍(Tan phung) 기자] 팜 민 찐(Pham Min...",http://www.insidevina.com/news/articleView.htm...,http://www.insidevina.com/news/thumbnail/20240...,2024-07-04T04:24:07Z,"[=, (Tan phung) ] (Pham Minh Chinh) 3 , .\r\n...",None,Insidevina.com
6,윤수현 기자,"넷플릭스, 베이직 요금제 완전 폐지 수순 - 미디어오늘",넷플릭스가 캐나다·영국에서 베이직 요금제를 완전 폐지하기로 했다. 베이직 요금제 유...,https://www.mediatoday.co.kr/news/articleView....,https://cdn.mediatoday.co.kr/news/thumbnail/20...,2024-07-04T04:11:30Z,· . .\r\n IT 2 ( 9500) · “713 . ” . ( 5500) ( ...,None,Mediatoday.co.kr
7,None,"반도체장비 세계 1위 ASML, 화성에 1조원대 연구개발 시설 건립 - 한국무역협회","한국무역협회 무역 통상정보, 회원/업무지원, 무역통계, 협회안내 등 서비스 안내.",https://www.kita.net/board/totalTradeNews/tota...,https://kita.net/imgs/common/kita_logo_mid.png,2024-07-04T04:06:37Z,"1 ASML, 1 \r\n 'ASML- ' \r\n ASML ( ) () \r\n ...",None,Kita.net
8,차은지,편하게 살 빼려다 '실명' 될라…충격 연구결과 나왔다 - 한국경제,"편하게 살 빼려다 '실명' 될라…충격 연구결과 나왔다, ""위고비 복용시 '눈 뇌졸중...",https://www.hankyung.com/article/2024070419307,https://img.hankyung.com/photo/202407/99.34985...,2024-07-04T03:53:55Z,. /=\r\n.3() CNN · ' ' ''(NAION) .\r\nNAION ' ...,None,Hankyung.com
9,경향신문,가상자산사업자 불시 영업종료? 앞으론 한달전에 이용자 보호안 제출해야 - 경향신문,None,https://m.khan.co.kr/economy/finance/article/2...,None,2024-07-04T03:20:00Z,None,None,Khan.co.kr


In [35]:
json_data = json_data.drop(columns=['source.id','content'])


In [36]:
json_data

,author,title,description,url,urlToImage,publishedAt,source.name
0,김지숙,"삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스",삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼...,https://news.kbs.co.kr/news/view.do?ncd=8003727,http://news.kbs.co.kr/data/news/2024/07/04/202...,2024-07-04T06:20:00Z,Kbs.co.kr
1,정윤교 기자,"골드만삭스 ""한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화"" - 연합인포맥스",유력 투자은행 골드만삭스는 한국은행이 올해 8월 기준금리 인하에 나설 것으로 전망했...,https://news.einfomax.co.kr/news/articleView.h...,https://cdn.news.einfomax.co.kr/news/thumbnail...,2024-07-04T05:57:35Z,Einfomax.co.kr
2,안태호,10명한테 물어놓고 “10명 중 9명 합격”…에듀윌 과태료 500만원 - 한겨레,공무원 온라인 강의 서비스를 제공하는 에듀윌이 단 10명만 응답한 설문 결과를 근거...,https://www.hani.co.kr/arti/economy/economy_ge...,https://flexible.img.hani.co.kr/flexible/norma...,2024-07-04T05:22:00Z,Hani.co.kr
3,김기훈,서울 기후동행카드에 남양주시 참여…8월 별내·진접선 적용 - 연합뉴스,(서울=연합뉴스) 김기훈 기자 = 8월 지하철 8호선의 연장 별내선 개통에 맞춰 별...,https://www.yna.co.kr/view/AKR20240704018700004,https://img4.yna.co.kr/photo/yna/YH/2024/07/04...,2024-07-04T05:10:00Z,Yna.co.kr
4,이덕주,“빅테크 꼭지 신호인가?”...젠슨 황 이어 베이조스도 주식 매각 나선다 - 매일경제,최근 신고가를 경신하고 있는 주요 빅테크 기업들의 창업자들이 주식을 매각하면서 주가...,https://www.mk.co.kr/news/business/11058688,https://wimg-orig.mk.co.kr/news/cms/202407/04/...,2024-07-04T04:53:51Z,Mk.co.kr
5,떤 풍(Tan phung) 기자,"베트남 총리, 삼성전자 평택캠퍼스 방문…방한일정 마무리 - 인사이드비나","[인사이드비나=하노이, 떤 풍(Tan phung) 기자] 팜 민 찐(Pham Min...",http://www.insidevina.com/news/articleView.htm...,http://www.insidevina.com/news/thumbnail/20240...,2024-07-04T04:24:07Z,Insidevina.com
6,윤수현 기자,"넷플릭스, 베이직 요금제 완전 폐지 수순 - 미디어오늘",넷플릭스가 캐나다·영국에서 베이직 요금제를 완전 폐지하기로 했다. 베이직 요금제 유...,https://www.mediatoday.co.kr/news/articleView....,https://cdn.mediatoday.co.kr/news/thumbnail/20...,2024-07-04T04:11:30Z,Mediatoday.co.kr
7,None,"반도체장비 세계 1위 ASML, 화성에 1조원대 연구개발 시설 건립 - 한국무역협회","한국무역협회 무역 통상정보, 회원/업무지원, 무역통계, 협회안내 등 서비스 안내.",https://www.kita.net/board/totalTradeNews/tota...,https://kita.net/imgs/common/kita_logo_mid.png,2024-07-04T04:06:37Z,Kita.net
8,차은지,편하게 살 빼려다 '실명' 될라…충격 연구결과 나왔다 - 한국경제,"편하게 살 빼려다 '실명' 될라…충격 연구결과 나왔다, ""위고비 복용시 '눈 뇌졸중...",https://www.hankyung.com/article/2024070419307,https://img.hankyung.com/photo/202407/99.34985...,2024-07-04T03:53:55Z,Hankyung.com
9,경향신문,가상자산사업자 불시 영업종료? 앞으론 한달전에 이용자 보호안 제출해야 - 경향신문,None,https://m.khan.co.kr/economy/finance/article/2...,None,2024-07-04T03:20:00Z,Khan.co.kr


In [44]:
# 페이지 수 계산
num_pages = (total_articles + page_size - 1) // page_size

# 모든 기사를 저장할 리스트
all_articles = []

# 각 페이지에 대한 API 호출
for page in range(num_pages):
    # 페이지 번호를 포함한 URL 생성
    page_url = f'{url}&page={page + 1}'

    # API 호출
    response = requests.get(page_url)

    # 응답이 성공적인지 확인
    if response.status_code == 200:
        # 응답에서 기사 데이터 추출
        articles = json.loads(response.text)['articles']

        # content 열과 source.id 필드를 제거한 기사 리스트 생성
        for article in articles:
            if 'content' in article:
                del article['content']
            if 'source' in article and 'id' in article['source']:
                del article['source']['id']


        # 전체 기사 리스트에 추가
        all_articles.extend(articles)
    else:
        # 에러 처리
        print(f"Error fetching articles: {response.status_code}")
        break

# 파일을 저장할 경로 정의
file_path = '/content/drive/My Drive/all_articles.json'

# 전체 기사 리스트를 JSON 파일로 저장
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(all_articles, f, indent=4, ensure_ascii=False)

# 확인 메시지 출력
print(f"All articles saved to: {file_path}")


All articles saved to: /content/drive/My Drive/all_articles.json


In [57]:
import json
from google.colab import drive

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 파일 경로 정의
file_path = '/content/drive/My Drive/all_articles.json'

# JSON 파일의 내용 확인
with open(file_path, 'r', encoding='utf-8') as f:
    data = f.read()
    print(data[:1000])  # 처음 1000자만 출력하여 확인

    json_data = json.loads(data)
    df = pd.json_normalize(json_data)

# DataFrame 출력 (필요에 따라 출력 방법 변경 가능)
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[
    {
        "source": {
            "name": "Kbs.co.kr"
        },
        "author": "김지숙",
        "title": "삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스",
        "description": "삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼성전자 디바이스솔루션(DS) 부문은 오늘(4일) HBM 개발팀 신설을 골자로 하는 조직 개편을 실시했습니다.<br /> <br />메모리사업부 내에 있었던 HBM 전담 조직을 팀으로 신설한 것으로, 신임 팀장은 고성능 D램 제품 설계 전문가인 손영수 부사장이 맡기로 했습니다.<br /> <br />이 같은 HBM 개발팀 신설은 인공지능 시장 확대로 HBM 수요가 급증하면서, HB…",
        "url": "https://news.kbs.co.kr/news/view.do?ncd=8003727",
        "urlToImage": "http://news.kbs.co.kr/data/news/2024/07/04/20240704_qmlaaV.jpg",
        "publishedAt": "2024-07-04T06:20:00Z"
    },
    {
        "source": {
            "name": "Einfomax.co.kr"
        },
        "author": "정윤교 기자",
        "title": "골드만삭스 \"한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화\" - 연합인포맥스",
        "description": "유력 투자은행 골드만삭스는 한국은행이

,author,title,description,url,urlToImage,publishedAt,source.name
0,김지숙,"삼성전자, 조직 개편 'HBM 개발팀' 신설…“HBM에 더 집중” - KBS뉴스",삼성전자가 HBM(고대역폭 메모리) 개발팀을 신설했습니다.<br /> <br />삼...,https://news.kbs.co.kr/news/view.do?ncd=8003727,http://news.kbs.co.kr/data/news/2024/07/04/202...,2024-07-04T06:20:00Z,Kbs.co.kr
1,정윤교 기자,"골드만삭스 ""한은 8월 금리 인하할 것…하반기 물가 2.0%로 둔화"" - 연합인포맥스",유력 투자은행 골드만삭스는 한국은행이 올해 8월 기준금리 인하에 나설 것으로 전망했...,https://news.einfomax.co.kr/news/articleView.h...,https://cdn.news.einfomax.co.kr/news/thumbnail...,2024-07-04T05:57:35Z,Einfomax.co.kr
2,안태호,10명한테 물어놓고 “10명 중 9명 합격”…에듀윌 과태료 500만원 - 한겨레,공무원 온라인 강의 서비스를 제공하는 에듀윌이 단 10명만 응답한 설문 결과를 근거...,https://www.hani.co.kr/arti/economy/economy_ge...,https://flexible.img.hani.co.kr/flexible/norma...,2024-07-04T05:22:00Z,Hani.co.kr
3,김기훈,서울 기후동행카드에 남양주시 참여…8월 별내·진접선 적용 - 연합뉴스,(서울=연합뉴스) 김기훈 기자 = 8월 지하철 8호선의 연장 별내선 개통에 맞춰 별...,https://www.yna.co.kr/view/AKR20240704018700004,https://img4.yna.co.kr/photo/yna/YH/2024/07/04...,2024-07-04T05:10:00Z,Yna.co.kr
4,이덕주,“빅테크 꼭지 신호인가?”...젠슨 황 이어 베이조스도 주식 매각 나선다 - 매일경제,최근 신고가를 경신하고 있는 주요 빅테크 기업들의 창업자들이 주식을 매각하면서 주가...,https://www.mk.co.kr/news/business/11058688,https://wimg-orig.mk.co.kr/news/cms/202407/04/...,2024-07-04T04:53:51Z,Mk.co.kr
...,...,...,...,...,...,...,...
65,박윤수,'해외 원정 성형' 의사‥역외탈세 무더기 적발 - MBC 뉴스,갈수록 치밀해지는 역외 탈세 수법에 금융당국이 적발에 나섰습니다. 동남아 원정 진료...,https://imnews.imbc.com/replay/2024/nwtoday/ar...,https://image.imnews.imbc.com/replay/2024/nwto...,2024-07-02T21:52:08Z,Imbc.com
66,박해윤 기자,청사진 공개된 롯바 송도 바이오캠퍼스…미국 시러큐스 캠퍼스 연계로 글로벌 시장 공략...,롯데바이오로직스가 인천 송도국제도시에 들어서는 36만ℓ 규모 바이오 캠퍼스와 미국 ...,https://www.incheonilbo.com/news/articleView.h...,https://www.incheonilbo.com/news/thumbnail/202...,2024-07-02T21:00:00Z,Incheonilbo.com
67,강동헌,"""금리 인하땐 제조업 AI투자 활발해져…반도체, 전력설비 등 주목"" - 서울경제신문",증권 > 증권일반 뉴스: 투자 전문가들은 서울경제신문이 2일 주최한 ‘머니트렌드 2...,https://www.sedaily.com/NewsView/2DBLI4LCJ3,https://newsimg.sedaily.com/2024/07/03/2DBLI4L...,2024-07-02T21:00:00Z,Sedaily.com
68,한지훈 기자,글로벌 진격... '퍼스트 디센던트' 3시간 만에 동접 17만 돌파 - 게임플,‘퍼스트 디센던트’에 대한 글로벌 게이머들의 관심이 엄청나다. 출시 당일 단 3시간...,https://www.gameple.co.kr/news/articleView.htm...,https://cdn.gameple.co.kr/news/photo/202407/21...,2024-07-02T11:11:34Z,Gameple.co.kr


No charts were generated by quickchart


In [45]:
# JSON 응답을 로드합니다.

data = response.json()

# 데이터프레임으로 변환합니다.
df = pd.json_normalize(data['articles'])

# 데이터프레임을 출력합니다.
print(df)


   author                                              title  \
0     민수정     "이게 5000원짜리 백반" 반찬만 13개…노부부 식당에 뜨거운 반응 - 머니투데이   
1     이정현             테슬라, 차량 인도량 전망치 상회…주가 10% 급등 - ZD넷 코리아   
2    None     HLB, FDA와 공장실사 보완 위한 미팅 완료…“추가 보완사항 없다” - 더바이오   
3     이소은    "송파구보다 비싸"…서울 아닌데 '평균 아파트값 16.3억' 찍은 이곳 - 머니투데이   
4     홍장원  일론 머스크 “테슬라 공매도 파멸할 것, 그게 빌 게이츠라도 마찬가지” [월가월부]...   
5     박윤수                 '해외 원정 성형' 의사‥역외탈세 무더기 적발 - MBC 뉴스   
6  박해윤 기자  청사진 공개된 롯바 송도 바이오캠퍼스…미국 시러큐스 캠퍼스 연계로 글로벌 시장 공략...   
7     강동헌     "금리 인하땐 제조업 AI투자 활발해져…반도체, 전력설비 등 주목" - 서울경제신문   
8  한지훈 기자        글로벌 진격... '퍼스트 디센던트' 3시간 만에 동접 17만 돌파 - 게임플   
9     박주희              대구서 월급 꼬박 모아 아파트 사는 데 걸리는 시간은? - 영남일보   

                                         description  \
0  푸짐한 한 끼를 제공하는 부산의 한 백반집이 화제다. 밥과 국을 제외하고 반찬 13...   
1  테슬라의 2분기 자동차 생산량과 인도량이 예상을 크게 넘어선 것으로 나타났다. 이 ...   
2                                               None   
3  '준강남'으로 불리는 경기도 과천시 평균 아파트 매맷값이 서울 비강남권은 물론, 분...   
4  2일(현지시간) 미국 